# Finding patients who have more than one kind of data

*Standalone -- no prior vignette required.*

> **DRAFT STATUS:** specific counts below are placeholders -- not yet run against the live API. Structure is the reviewed part; content needs verification.

I don't just want "everyone with kidney cancer" -- I want people who have *both* imaging and sequencing data for the same kidney cancer, because my actual hypothesis needs to compare across modalities for the same person, not just pool two separate populations.

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]

First, what file types actually exist to ask for:

In [ ]:
from cdapython import columns, column_values
columns(table='file')

In [ ]:
column_values('file_type')

`CT Image Storage` and `Annotated Somatic Mutation` look like the two I want. If I put both in one `match_all`, though, that won't work the way I want -- `match_all` means every condition has to be true of the *same row*, and no single file can be both a CT image and a mutation file at once. What I actually need is two separate searches -- kidney subjects with CT data, kidney subjects with mutation data -- and then the overlap between them.

In [ ]:
from cdapython import get_subject_data

ct_subjects = get_subject_data(
    match_all=['anatomic_site = *kidney*', 'file_type = CT Image Storage'],
    add_columns='file.*'
)

mutation_subjects = get_subject_data(
    match_all=['anatomic_site = *kidney*', 'file_type = Annotated Somatic Mutation'],
    add_columns='file.*'
)

`add_columns='file.*'` pulls in every column from the file table so I can see where to actually get these files later -- that's a lot of columns I don't all need, so I can drop ones I know I won't use:

In [ ]:
ct_subjects = get_subject_data(
    match_all=['anatomic_site = *kidney*', 'file_type = CT Image Storage'],
    add_columns='file.*',
    exclude_columns='checksum'
)

Now, the actual overlap -- subjects appearing in *both* result sets:

In [ ]:
from cdapython import intersect_subject_results

ct_and_mutation = intersect_subject_results(ct_subjects, mutation_subjects)
ct_and_mutation

That's my actual cohort -- people who have both. But looking at the `file_type` and `file_id` columns, I can't tell which file belongs to which type for a given subject; everything's mashed together in one row per subject. Re-running with `collate_results=True` pairs up the related file columns per file, so I can tell which files have which properties:

In [ ]:
ct_subjects = get_subject_data(
    match_all=['anatomic_site = *kidney*', 'file_type = CT Image Storage'],
    add_columns='file.*',
    collate_results=True
)
mutation_subjects = get_subject_data(
    match_all=['anatomic_site = *kidney*', 'file_type = Annotated Somatic Mutation'],
    add_columns='file.*',
    collate_results=True
)
ct_and_mutation = intersect_subject_results(ct_subjects, mutation_subjects)
ct_and_mutation

There's a new `file_data` column now, with `data_source`, `file_id`, and `access` collated per file -- useful for feeding into another function, but still a nested structure that's hard to just look at directly. `expand_subject_results` flattens it into a normal table, one row per file instead of one row per subject:

In [ ]:
from cdapython import expand_subject_results
expand_subject_results(ct_and_mutation, 'file_data')

Same subjects as before, but now anyone with multiple files gets multiple rows, and each row only shows the values that actually go together. I now have exactly the cohort I needed -- confirmed to have both modalities, with the files per person untangled instead of guessed at -- without ever needing to know how the underlying tables related to each other before I started.